In [1]:
# Baseline Eigenmode Simulation (Transmon Only)
from pathlib import Path
from qiskit_metal import designs, MetalGUI
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket
from qiskit_metal.toolbox_metal.parsing import parse_units
from qiskit_metal.renderers.renderer_qtcad.qtcad_renderer import QQTCADRenderer
import numpy as np

In [2]:
# --- PARAMETER SWEEP SETTINGS ---
width_adjustment = 0
height_adjustment = 0

pad_width_total = f"{500 + width_adjustment}um"
pad_height_total = f"{215 + height_adjustment}um"

print(f"Sweep Params: Width={pad_width_total}, Height={pad_height_total}")

# Set up directory
device_name = "transmon_baseline_eig"
output_dir = str(Path.cwd() / "output" / device_name)
geo_filepath = output_dir + "/" + device_name + ".xao"
mesh_filepath = output_dir + "/" + device_name + ".msh4"

Sweep Params: Width=500um, Height=215um


In [3]:
tolerance_relative = 0.1
mesh_h_min = "150um"
mesh_h_max = "150um"
num_modes = 2

In [4]:
design = designs.MultiPlanar({}, True)
design.overwrite_enabled = True

transmon_label = "Q1"

# Set up transmon pocket with connection pad for baseline
options_qubit = dict(
    pos_x='0um',
    pos_y='0um',
    pad_gap='30um',
    pad_width=pad_width_total,
    pad_height=pad_height_total,
    pocket_width='700um',
    pocket_height='700um',
    orientation="0",
    connection_pads=dict(
        coupler=dict(loc_W=1, loc_H=-1, pad_width='70um', cpw_extend='50um')
    )
)

qubit = TransmonPocket(design, transmon_label, options=options_qubit)

# Mark the coupler pin as open for the simulation
open_pins = [(transmon_label, 'coupler')]

In [5]:
gui = MetalGUI(design)
gui.rebuild()
gui.autoscale()

In [ ]:
gui.main_window.close()

In [ ]:
# --- QTCAD EIGENMODE SIMULATION ---

# 1. Define physical constants as pure numbers
inductance_val = 10e-9
gap_val = float(parse_units(qubit.options.pad_gap))
print(f"Using numeric values: Lj={inductance_val} H, Gap={gap_val} m")

# 2. Initialize Renderer
qtcad_renderer = QQTCADRenderer(
    design,
    options=dict(
        adaptive=True,
        output_dir=output_dir,
        geo_filepath=geo_filepath,
        mesh_filepath=mesh_filepath,
        mesh_scale=1e-3,
        adaptive_mesh_scale=1e-3,
        make_subdir=False,
        maxwell_emode=dict(num_modes=num_modes, tol_rel=tolerance_relative),
    ),
)

# 3. Render and Setup Junction
qtcad_renderer.render_design(
    open_pins=[],
    skip_junctions=False,
    mesh_geoms=False,
    initial_mesh_h_min=mesh_h_min,
    initial_mesh_h_max=mesh_h_max,
)
qtcad_renderer.set_up_junction(transmon_label, inductance_val, gap_val)

# 4. Mesh and Solve
print("Meshing device...")
qtcad_renderer.gmsh.add_mesh(dim=3, intelli_mesh=False)
qtcad_renderer.export_mesh()
qtcad_renderer.export_parameters()

print("\n--- Running QTCAD Maxwell Eigenmode Simulation ---")
qtcad_renderer.run_qtcad("eigs")

# 5. Load Results
frequencies = qtcad_renderer.load_qtcad_maxwell_eigenmodes()
print("\nMaxwell Eigenmode Frequencies (GHz):")
print(frequencies)

qtcad_renderer.plot_eigenmodes()

Using numeric values: Lj=1e-08 H, Gap=2.9999999999999997e-05 m
Meshing device...

--- Running QTCAD Maxwell Eigenmode Simulation ---


09:52AM 42s INFO [run_qtcad]: =================
09:52AM 42s INFO [run_qtcad]: Running QTCAD®...
09:52AM 42s INFO [run_qtcad]: =================



|  ||  |||                                                                 \\  
|  ||  |||         @@@@     @@@@@@@@      @@@@@        @       @@@@@@@      \\ 
|  ||  |||      @@@   @@@      @@      @@@    @       @@       @@    @@@     \\
|  ||  |||     @@       @@     @@     @@             @ @@      @@     @@@     \\
|  ||  |||     @@       @@     @@     @@            @    @     @@      @@     //
|  ||  |||      @@     @@      @@      @@@    @    @@@@@@@@    @@    @@@     //
|  ||  |||        @@@@@@       @@        @@@@@@   @@      @@   @@@@@@       // 
|  ||  |||             @@                                                  //   

                                 Version 2.1.3                                  
  Copyright (c) 2022-2026 Nanoacademic Technologies Inc. All rights reserved.   

      Welcome to QTCAD, the Quantum-Technology Computer-Aided Design tool.      

                        For documentation, please visit:                        
                      https:/